In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# Restart runtime after this cell
# ============================================================
!pip install -q transformers peft accelerate bitsandbytes datasets trl scikit-learn openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 46.4 MB/s eta 0:00:00


In [4]:
# ============================================================
# CELL 2 — MOUNT DRIVE AND CONFIGURATION
# ============================================================
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

# ---- OpenAI (optional — only for ACCEPT reasoning enhancement in Cell 10) ----
# If you skip Cell 10, this is unused
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# ---- Paths ----
DRIVE_ROOT   = '/content/drive/MyDrive/298B_WB2'
RAW_PRS_CSV  = f'{DRIVE_ROOT}/raw_prs.csv'
OUTPUT_DIR   = f'{DRIVE_ROOT}/training_run_critic'
ADAPTER_DIR  = f'{DRIVE_ROOT}/critic_lora_adapter'
JSONL_PATH   = f'{DRIVE_ROOT}/critic_training.jsonl'
CSV_BACKUP   = f'{DRIVE_ROOT}/critic_pairs.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

# ---- Model ----
BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'

# ---- Balance config ----
# EQUAL 33/33/33 balance — strongest for learning ACCEPT vs REVISE vs REJECT boundary
# Cap is determined by how many clean REJECT examples we can extract
# from the 527 not-merged PRs (those with review criticism)
# After Cell 5 runs, PER_CLASS_TARGET auto-adjusts to available REJECT count
PER_CLASS_TARGET = 192   # hard cap = clean REJECT examples available (192 from 527 not-merged PRs)

# ---- Diff truncation ----
MAX_DIFF_TOKENS = 400   # first hunk per file, total cap across all files

# ---- LoRA config ----
LORA_R           = 32
LORA_ALPHA       = 64
LORA_DROPOUT     = 0.05
LORA_TARGET_MODS = ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj']

# ---- Training hyperparameters ----
# A100 80GB — batch 4 because Critic inputs include diffs (longer than Planner)
# No weighted loss — equal balance means standard CE is sufficient
MAX_SEQ_LEN  = 2048
EPOCHS       = 6    # early stopping will terminate earlier if plateau detected
BATCH_SIZE   = 4
GRAD_ACCUM   = 4      # effective batch = 16
LR           = 1e-4
WARMUP_STEPS = 15
WEIGHT_DECAY = 0.01

SEED = 42
print('Config loaded.')
print(f'Equal balance: {PER_CLASS_TARGET} per class = {PER_CLASS_TARGET * 3} total')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config loaded.
Equal balance: 192 per class = 576 total


In [5]:
# ============================================================
# CELL 3 — IMPORTS
# ============================================================
import json
import re
import csv as csv_module
import random
import time
import warnings
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
random.seed(SEED)
csv_module.field_size_limit(10 * 1024 * 1024)

print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
assert Path(RAW_PRS_CSV).exists(), f'raw_prs.csv not found at {RAW_PRS_CSV}'

PyTorch 2.10.0+cu128 | CUDA True
GPU: NVIDIA A100-SXM4-80GB


In [6]:
# ============================================================
# CELL 4 — LOAD raw_prs.csv AND SPLIT INTO MERGED / NOT-MERGED
# ============================================================
import pandas as pd
import csv as csv_module
csv_module.field_size_limit(10 * 1024 * 1024)

# Update path to CSV
RAW_PRS_CSV = f'{DRIVE_ROOT}/raw_prs.csv'
assert Path(RAW_PRS_CSV).exists(), f'raw_prs.csv not found at {RAW_PRS_CSV}'

print(f'Loading {RAW_PRS_CSV}...')
prs_df = pd.read_csv(RAW_PRS_CSV)
prs_df.columns = [c.upper() for c in prs_df.columns]
print(f'Total PRs loaded: {len(prs_df)}')
print(f'Columns: {list(prs_df.columns)}')

# Convert each row into a dict matching the JSON record structure
# RAW_JSON column contains the full nested JSON per PR
all_prs = []
for _, row in prs_df.iterrows():
    try:
        raw = json.loads(row['RAW_JSON'])
        # Inject scalar columns in case raw_json is missing them
        raw['pr_number']  = row.get('PR_NUMBER')
        raw['is_merged']  = row.get('IS_MERGED', False)
        all_prs.append(raw)
    except Exception as e:
        continue

print(f'Successfully parsed: {len(all_prs)} PR records')

merged_prs     = []
not_merged_prs = []

for record in all_prs:
    pr        = record.get('pr', record)
    is_merged = (pr.get('merged_at') is not None) or record.get('is_merged', False)
    if is_merged:
        merged_prs.append(record)
    else:
        not_merged_prs.append(record)

print(f'Merged (ACCEPT/REVISE source):  {len(merged_prs)}')
print(f'Not merged (REJECT source):     {len(not_merged_prs)}')

Loading /content/drive/MyDrive/298B_WB2/raw_prs.csv...
Total PRs loaded: 2802
Columns: ['PR_NUMBER', 'REPO', 'LINKED_ISSUE_NUMBER', 'PR_TITLE', 'STATE', 'IS_MERGED', 'BASE_SHA', 'HEAD_SHA', 'CHANGED_FILES_COUNT', 'REVIEW_COUNT', 'CI_CONCLUSION', 'MERGED_AT', 'CREATED_AT', 'EXTRACTED_AT', 'RAW_JSON']
Successfully parsed: 2802 PR records
Merged (ACCEPT/REVISE source):  2178
Not merged (REJECT source):     624


In [7]:
# ============================================================
# CELL 5 — HELPERS: DIFF EXTRACTION AND PAIR BUILDERS
# ============================================================

def extract_first_hunk(patch: str) -> str:
    """Extract only the first @@ hunk from a patch. Discard everything after second @@."""
    if not patch:
        return ''
    lines      = patch.split('\n')
    result     = []
    hunk_count = 0
    for line in lines:
        if line.startswith('@@'):
            hunk_count += 1
            if hunk_count == 1:
                result.append(line)
            else:
                break
        elif hunk_count == 1:
            result.append(line)
    return '\n'.join(result)


def extract_diff(files: list, max_tokens: int = MAX_DIFF_TOKENS) -> str:
    """First hunk per file, total capped at max_tokens."""
    parts      = []
    used_chars = 0
    max_chars  = max_tokens * 4
    for f in files:
        hunk = extract_first_hunk(f.get('patch', ''))
        if not hunk:
            continue
        entry = f"{f.get('filename', '')}:\n{hunk}"
        if used_chars + len(entry) > max_chars:
            break
        parts.append(entry)
        used_chars += len(entry)
    return '\n\n'.join(parts)


def trunc(text: str, max_tokens: int) -> str:
    mc = max_tokens * 4
    return text[:mc] + '...' if len(text) > mc else text


def build_input(record: dict) -> str:
    """
    Build Critic input.
    At training: PLAN = pr.body (proxy for Planner output)
    At inference: PLAN = actual Planner model output
    Format is identical — model generalises across the two sources.
    """
    pr    = record.get('pr', record)
    files = record.get('files', [])
    rcs   = record.get('review_comments', [])

    plan     = trunc((pr.get('body') or '').strip().replace('\n', ' '), 200)
    diff     = extract_diff(files)
    rc_parts = [
        (c.get('body') or '').strip()[:200]
        for c in rcs
        if len((c.get('body') or '').strip()) > 20
    ][:3]
    review_ctx = ' | '.join(rc_parts) if rc_parts else 'none'

    return (
        'You are a senior code reviewer evaluating a proposed patch for an Apache Airflow issue.\n'
        'Output ACCEPT, REVISE, or REJECT with specific reasoning.\n\n'
        f'PLAN: {plan or "No description provided."}\n\n'
        f'DIFF:\n{diff}\n\n'
        f'REVIEW_CONTEXT: {review_ctx}\n\n'
        'Evaluate the patch and output your verdict.'
    )


def build_output(label: str, reasoning: str) -> str:
    return f'VERDICT: {label}\nREASONING: {reasoning}'


def build_full_text(inp: str, out: str) -> str:
    return f'<|im_start|>user\n{inp}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>'


# Sanity test
test_patch = '@@ -10,5 +10,7 @@\n ctx\n-old\n+new\n@@ -50,3 +52,3 @@\n other'
assert '@@ -50' not in extract_first_hunk(test_patch)
print('Helpers loaded. First hunk extraction: PASS')

Helpers loaded. First hunk extraction: PASS


In [8]:
# ============================================================
# CELL 6 — LOAD TOKENIZER
# ============================================================
print(f'Loading tokenizer: {BASE_MODEL}')
TOKENIZER = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
TOKENIZER.pad_token    = TOKENIZER.eos_token
TOKENIZER.padding_side = 'right'

def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text, add_special_tokens=False))

print('Tokenizer ready.')

Loading tokenizer: Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.


In [9]:
# ============================================================
# CELL 7 — BUILD REJECT PAIRS FIRST
#
# Why build REJECT first:
#   REJECT is the bottleneck — 527 not-merged PRs but only a subset
#   have usable review criticism. We build REJECT first to know
#   the actual cap, then match ACCEPT and REVISE to it.
#
# REJECT reasoning source:
#   Priority 1: CHANGES_REQUESTED review body (explicit rejection reason)
#   Priority 2: Substantive review_comments with diff_hunk
#   Skip: PRs with no review signal — not abandoned PRs, just no explanation
#          Never fabricate a reason. Model learns nothing from a made-up REJECT reason.
# ============================================================

reject_pairs  = []
reject_skip   = 0
random.shuffle(not_merged_prs)

for record in not_merged_prs:
    reviews = record.get('reviews', [])
    rcs     = record.get('review_comments', [])
    files   = record.get('files', [])

    if not any(f.get('patch') for f in files):
        reject_skip += 1
        continue

    # Priority 1: CHANGES_REQUESTED with body
    cr_reviews = [
        r for r in reviews
        if r.get('state') == 'CHANGES_REQUESTED'
        and len((r.get('body') or '').strip()) > 20
    ]

    # Priority 2: Inline review comments with diff context
    sub_comments = [
        c for c in rcs
        if c.get('diff_hunk')
        and len((c.get('body') or '').strip()) > 20
        and not (c.get('body', '').strip().startswith('```suggestion'))
    ]

    if cr_reviews:
        reasoning = cr_reviews[0].get('body', '').strip()[:400]
    elif sub_comments:
        reasoning = sub_comments[0].get('body', '').strip()[:400]
    else:
        # No usable review signal — skip entirely
        # Do NOT fabricate reasoning
        reject_skip += 1
        continue

    inp = build_input(record)
    out = build_output('REJECT', reasoning)
    ft  = build_full_text(inp, out)
    tc  = count_tokens(ft)

    if 50 < tc <= MAX_SEQ_LEN:
        reject_pairs.append({
            'pr_number':   record.get('pr_number', record.get('pr', {}).get('number')),
            'label':       'REJECT',
            'confidence':  'high',
            'input':       inp, 'output': out, 'full_text': ft, 'token_count': tc,
        })
    else:
        reject_skip += 1

# This number determines the equal-balance cap for all three classes
ACTUAL_CAP = min(len(reject_pairs), PER_CLASS_TARGET)
reject_pairs = reject_pairs[:ACTUAL_CAP]

print(f'Clean REJECT pairs available: {len(reject_pairs) + (len(reject_pairs) - ACTUAL_CAP)} (before cap)')
print(f'REJECT pairs used: {len(reject_pairs)} (capped at {ACTUAL_CAP})')
print(f'Skipped (no review signal or no diff): {reject_skip}')
print(f'\n→ Equal balance cap set to: {ACTUAL_CAP} per class')
print(f'→ Total training pairs target: {ACTUAL_CAP * 3}')

Clean REJECT pairs available: 192 (before cap)
REJECT pairs used: 192 (capped at 192)
Skipped (no review signal or no diff): 384

→ Equal balance cap set to: 192 per class
→ Total training pairs target: 576


In [10]:
# ============================================================
# CELL 8 — BUILD REVISE PAIRS FROM MERGED PRs
#
# REVISE = merged PR that had CHANGES_REQUESTED or inline criticism
# Even though it was eventually merged, the intermediate criticism
# IS the REVISE signal — 'here's what needed fixing before acceptance'
# Capped at ACTUAL_CAP (equal balance with REJECT)
#
# Reasoning source:
#   Priority 1: Inline review_comment with diff_hunk (most specific)
#   Priority 2: CHANGES_REQUESTED review body
# ============================================================

revise_pairs = []
revise_skip  = 0
random.shuffle(merged_prs)

for record in merged_prs:
    if len(revise_pairs) >= ACTUAL_CAP:
        break

    reviews = record.get('reviews', [])
    rcs     = record.get('review_comments', [])
    files   = record.get('files', [])

    if not any(f.get('patch') for f in files):
        revise_skip += 1
        continue

    cr_reviews = [
        r for r in reviews
        if r.get('state') == 'CHANGES_REQUESTED'
        and len((r.get('body') or '').strip()) > 20
    ]
    sub_comments = [
        c for c in rcs
        if c.get('diff_hunk')
        and len((c.get('body') or '').strip()) > 20
        and not c.get('body', '').strip().startswith('```suggestion')
    ]

    if not (cr_reviews or sub_comments):
        continue  # no REVISE signal — this PR goes to ACCEPT pool

    # Prefer inline comment (more specific) over CHANGES_REQUESTED body
    if sub_comments:
        reasoning = sub_comments[0].get('body', '').strip()[:400]
    else:
        reasoning = cr_reviews[0].get('body', '').strip()[:400]

    inp = build_input(record)
    out = build_output('REVISE', reasoning)
    ft  = build_full_text(inp, out)
    tc  = count_tokens(ft)

    if 50 < tc <= MAX_SEQ_LEN:
        revise_pairs.append({
            'pr_number':   record.get('pr_number', record.get('pr', {}).get('number')),
            'label':       'REVISE',
            'confidence':  'high',
            'input':       inp, 'output': out, 'full_text': ft, 'token_count': tc,
        })
    else:
        revise_skip += 1

print(f'REVISE pairs: {len(revise_pairs)} (target: {ACTUAL_CAP})')
if len(revise_pairs) < ACTUAL_CAP:
    print(f'WARNING: Only {len(revise_pairs)} REVISE — your Snowflake query found 757, check JSON structure')

REVISE pairs: 192 (target: 192)


In [11]:
# ============================================================
# CELL 9 — BUILD ACCEPT PAIRS FROM MERGED PRs
#
# ACCEPT = merged PR with only APPROVED reviews, no CHANGES_REQUESTED,
# no substantive inline comments, no failed key CI checks
#
# REASONING for ACCEPT:
#   The absence of criticism IS the signal — but the model needs to output
#   something meaningful, not just 'looks good'.
#   Strategy:
#     If an APPROVED review has a body (even 'LGTM, clean implementation'),
#     use that as reasoning.
#     Otherwise use a templated observation based on diff characteristics:
#     - Single file change → 'Targeted single-file fix with minimal footprint'
#     - Few files, no test changes → 'Clean patch addressing the described issue'
#     - Test files present → 'Patch includes test coverage, no review objections'
#   This teaches the model to reference diff characteristics in ACCEPT verdicts
#   rather than outputting a generic phrase every time.
# ============================================================

CI_FAIL = {'failure', 'timed_out', 'action_required'}
KEY_CI  = {'static checks', 'tests', 'ci/cd', 'build', 'lint', 'mypy'}

accept_pairs = []
accept_skip  = 0

# Shuffle merged_prs again to get a fresh random order for ACCEPT selection
random.shuffle(merged_prs)

for record in merged_prs:
    if len(accept_pairs) >= ACTUAL_CAP:
        break

    reviews    = record.get('reviews', [])
    rcs        = record.get('review_comments', [])
    check_runs = record.get('check_runs', [])
    files      = record.get('files', [])

    if not any(f.get('patch') for f in files):
        accept_skip += 1
        continue

    # Exclude if any CHANGES_REQUESTED or substantive inline comments
    has_cr = any(
        r.get('state') == 'CHANGES_REQUESTED'
        for r in reviews
    )
    has_sub_comments = any(
        c.get('diff_hunk') and len((c.get('body') or '').strip()) > 20
        for c in rcs
    )
    if has_cr or has_sub_comments:
        continue  # this is a REVISE PR, not ACCEPT

    # Exclude if key CI checks failed
    failed_ci = any(
        c.get('conclusion') in CI_FAIL
        and any(k in (c.get('name') or '').lower() for k in KEY_CI)
        for c in check_runs
    )
    if failed_ci:
        accept_skip += 1
        continue

    # ---- Build reasoning ----
    # Try approved review body first
    approval_bodies = [
        (r.get('body') or '').strip()
        for r in reviews
        if r.get('state') == 'APPROVED'
        and len((r.get('body') or '').strip()) > 10
        # Exclude purely emoji or single-word approvals
        and not (r.get('body') or '').strip().lower() in {'lgtm', 'lgtm!', '+1', 'looks good', 'approved'}
    ]

    if approval_bodies:
        reasoning = approval_bodies[0][:300]
    else:
        # Derive reasoning from diff characteristics
        file_names = [f.get('filename', '') for f in files if f.get('patch')]
        has_tests  = any('test' in fn.lower() for fn in file_names)
        n_files    = len(file_names)

        if n_files == 1:
            reasoning = f'Targeted single-file patch to {file_names[0]}. Clean diff with no review objections and CI passing.'
        elif has_tests:
            reasoning = f'Patch modifies {n_files} files including test coverage. All reviewers approved with no requested changes.'
        else:
            reasoning = f'Clean patch across {n_files} files. No review objections, no CI failures. Diff addresses the described issue without unnecessary scope.'

    inp = build_input(record)
    out = build_output('ACCEPT', reasoning)
    ft  = build_full_text(inp, out)
    tc  = count_tokens(ft)

    if 50 < tc <= MAX_SEQ_LEN:
        accept_pairs.append({
            'pr_number':   record.get('pr_number', record.get('pr', {}).get('number')),
            'label':       'ACCEPT',
            'confidence':  'high',
            'input':       inp, 'output': out, 'full_text': ft, 'token_count': tc,
        })
    else:
        accept_skip += 1

print(f'ACCEPT pairs: {len(accept_pairs)} (target: {ACTUAL_CAP})')

ACCEPT pairs: 192 (target: 192)


In [12]:
# ============================================================
# CELL 10 — OPTIONAL: ENHANCE ACCEPT REASONING WITH GPT-4o-mini [Run if you dont have it persisted already, else run 10C]
#
# Skip this cell if you're happy with the diff-derived ACCEPT reasoning
# from Cell 9. Run it for higher quality ACCEPT reasoning (~$0.30).
#
# What it does: replaces the templated ACCEPT reasoning with a
# GPT-4o-mini generated 2-sentence observation about the diff quality.
# Makes ACCEPT outputs more varied and specific at inference.
#
# Rate limit protection:
#   - Catches openai.RateLimitError specifically (not string matching)
#   - Exponential backoff: 5s, 10s, 20s, 40s, 80s across 5 attempts
#   - Proactive 1.5s sleep between every call to avoid hitting TPM limit
#   - On exhaustion: keeps original templated reasoning, prints warning
# ============================================================
import openai

client = openai.OpenAI(api_key=OPENAI_API_KEY)

ACCEPT_ENHANCE_PROMPT = """You are writing a code review verdict for a patch that was ACCEPTED.
The patch was approved with no requested changes.

DIFF (first hunk per file):
{diff_text}

Write exactly 2 sentences explaining why this patch is acceptable:
- Be specific about what the patch does correctly (reference the actual changes)
- Do not invent issues that don't exist
- Write as if you are the reviewer who approved it

Output only the 2-sentence reasoning. No preamble."""


def enhance_accept_reasoning(diff_text: str, max_retries: int = 5) -> str | None:
    """Call GPT-4o-mini with exponential backoff on rate limits."""
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[{'role': 'user', 'content': ACCEPT_ENHANCE_PROMPT.format(diff_text=diff_text[:600])}],
                temperature=0.3,
                max_tokens=100,
            )
            return resp.choices[0].message.content.strip()
        except openai.RateLimitError:
            wait = (2 ** attempt) * 5 + random.uniform(0, 3)
            print(f'  Rate limit (attempt {attempt+1}/{max_retries}) — waiting {wait:.1f}s')
            time.sleep(wait)
        except openai.APIStatusError as e:
            if e.status_code == 429:
                wait = (2 ** attempt) * 5 + random.uniform(0, 3)
                print(f'  429 status (attempt {attempt+1}/{max_retries}) — waiting {wait:.1f}s')
                time.sleep(wait)
            else:
                print(f'  API error {e.status_code} — skipping this pair')
                return None
        except Exception as e:
            print(f'  Unexpected error — skipping: {e}')
            return None
    print(f'  Max retries exhausted — keeping original templated reasoning')
    return None


print(f'Enhancing ACCEPT reasoning for {len(accept_pairs)} pairs...')
enhanced = 0
failed   = 0

for i, pair in enumerate(accept_pairs):
    # Only enhance templated reasoning — approval body text is already good
    if any(trigger in pair['output'] for trigger in [
        'Targeted single-file', 'Clean patch across', 'modifies', 'no review objections'
    ]):
        diff_text  = pair['input'].split('DIFF:\n')[1].split('\n\nREVIEW_CONTEXT')[0]
        new_reason = enhance_accept_reasoning(diff_text)

        if new_reason and len(new_reason) > 20:
            pair['output']      = build_output('ACCEPT', new_reason)
            pair['full_text']   = build_full_text(pair['input'], pair['output'])
            pair['token_count'] = count_tokens(pair['full_text'])
            enhanced += 1
        else:
            failed += 1

        # Proactive pacing between every call — prevents hitting TPM limit
        time.sleep(1.5)

    if (i + 1) % 25 == 0:
        print(f'  Progress: {i+1}/{len(accept_pairs)} | enhanced={enhanced} failed={failed}')

print(f'\nDone. Enhanced: {enhanced} | Kept original: {failed}')

Enhancing ACCEPT reasoning for 192 pairs...
  Progress: 25/192 | enhanced=20 failed=0
  Progress: 50/192 | enhanced=43 failed=0
  Progress: 75/192 | enhanced=60 failed=0
  Progress: 100/192 | enhanced=78 failed=0
  Progress: 125/192 | enhanced=95 failed=0
  Progress: 150/192 | enhanced=114 failed=0
  Progress: 175/192 | enhanced=134 failed=0

Done. Enhanced: 146 | Kept original: 0


In [13]:
# ============================================================
# CELL 10B — PERSIST ENHANCED ACCEPT PAIRS TO DRIVE
# Run immediately after Cell 10 completes
# Saves all three label pools so you never need to rerun Cell 10
# and burn OpenAI credits again
# ============================================================
import json

LABELLED_PAIRS_PATH = f'{DRIVE_ROOT}/critic_labelled_pairs.json'

labelled_snapshot = {
    'accept_pairs': accept_pairs,
    'revise_pairs': revise_pairs,
    'reject_pairs': reject_pairs,
    'actual_cap':   ACTUAL_CAP,
}

with open(LABELLED_PAIRS_PATH, 'w') as f:
    json.dump(labelled_snapshot, f, indent=2)

size_mb = Path(LABELLED_PAIRS_PATH).stat().st_size / (1024 * 1024)
print(f'Labelled pairs saved to Drive: {LABELLED_PAIRS_PATH} ({size_mb:.1f}MB)')
print(f'  ACCEPT: {len(accept_pairs)} | REVISE: {len(revise_pairs)} | REJECT: {len(reject_pairs)}')
print(f'  ACTUAL_CAP: {ACTUAL_CAP}')
print('Next time load from this file — skip Cells 7-10 entirely.')

Labelled pairs saved to Drive: /content/drive/MyDrive/298B_WB2/critic_labelled_pairs.json (2.6MB)
  ACCEPT: 192 | REVISE: 192 | REJECT: 192
  ACTUAL_CAP: 192
Next time load from this file — skip Cells 7-10 entirely.


In [ ]:
# ============================================================
# CELL 10C — RELOAD LABELLED PAIRS FROM DRIVE (skip Cells 7-10)
# Run this instead of Cells 7-10 in future sessions
# ============================================================
LABELLED_PAIRS_PATH = f'{DRIVE_ROOT}/critic_labelled_pairs.json'

with open(LABELLED_PAIRS_PATH) as f:
    snapshot = json.load(f)

accept_pairs = snapshot['accept_pairs']
revise_pairs = snapshot['revise_pairs']
reject_pairs = snapshot['reject_pairs']
ACTUAL_CAP   = snapshot['actual_cap']

print(f'Loaded from Drive: {LABELLED_PAIRS_PATH}')
print(f'  ACCEPT: {len(accept_pairs)} | REVISE: {len(revise_pairs)} | REJECT: {len(reject_pairs)}')
print(f'  ACTUAL_CAP: {ACTUAL_CAP}')
print('Proceed directly to Cell 11.')

In [14]:
# ============================================================
# CELL 11 — ASSEMBLE, VALIDATE AND SAVE
# ============================================================
import csv as csv_builtin

all_pairs = accept_pairs[:ACTUAL_CAP] + revise_pairs[:ACTUAL_CAP] + reject_pairs[:ACTUAL_CAP]
random.shuffle(all_pairs)

n_a = sum(1 for p in all_pairs if p['label'] == 'ACCEPT')
n_r = sum(1 for p in all_pairs if p['label'] == 'REVISE')
n_j = sum(1 for p in all_pairs if p['label'] == 'REJECT')
total = len(all_pairs)

print(f'Total: {total}')
print(f'  ACCEPT: {n_a} ({n_a/total*100:.1f}%)')
print(f'  REVISE: {n_r} ({n_r/total*100:.1f}%)')
print(f'  REJECT: {n_j} ({n_j/total*100:.1f}%)')

tc_sorted = sorted(p['token_count'] for p in all_pairs)
print(f'\nTokens: min={tc_sorted[0]} median={tc_sorted[len(tc_sorted)//2]} '
      f'p90={tc_sorted[int(len(tc_sorted)*0.9)]} max={tc_sorted[-1]}')

over = [p['pr_number'] for p in all_pairs if p['token_count'] > MAX_SEQ_LEN]
assert len(over) == 0, f'Token overflow: {over[:3]}'
print('Token limit check: PASS')

# Spot check — print one sample per label
for lbl in ['ACCEPT', 'REVISE', 'REJECT']:
    s = next((p for p in all_pairs if p['label'] == lbl), None)
    if s:
        print(f'\nSAMPLE {lbl} — PR #{s["pr_number"]} | {s["token_count"]}t')
        print('OUTPUT:', s['output'][:250])

# Save JSONL
with open(JSONL_PATH, 'w') as f:
    for p in all_pairs:
        f.write(json.dumps({'text': p['full_text']}) + '\n')
print(f'\nJSONL saved: {JSONL_PATH}')

# Save CSV backup
with open(CSV_BACKUP, 'w', newline='') as f:
    w = csv_builtin.DictWriter(f, fieldnames=[
        'pr_number','label','confidence','token_count','input_preview','output'
    ])
    w.writeheader()
    for p in all_pairs:
        w.writerow({
            'pr_number':     p['pr_number'],
            'label':         p['label'],
            'confidence':    p['confidence'],
            'token_count':   p['token_count'],
            'input_preview': p['input'][:300].replace('\n', ' '),
            'output':        p['output'].replace('\n', ' | '),
        })
print(f'CSV backup saved: {CSV_BACKUP}')

Total: 576
  ACCEPT: 192 (33.3%)
  REVISE: 192 (33.3%)
  REJECT: 192 (33.3%)

Tokens: min=138 median=539 p90=754 max=1049
Token limit check: PASS

SAMPLE ACCEPT — PR #63494 | 421t
OUTPUT: VERDICT: ACCEPT
REASONING: Looks fine to me

SAMPLE REVISE — PR #60728 | 358t
OUTPUT: VERDICT: REVISE
REASONING: This function is not called anywhere? Or do I miss it?

SAMPLE REJECT — PR #63454 | 568t
OUTPUT: VERDICT: REJECT
REASONING: Please do me a favor and drop this comment while you are in here.
```suggestion

```

JSONL saved: /content/drive/MyDrive/298B_WB2/critic_training.jsonl
CSV backup saved: /content/drive/MyDrive/298B_WB2/critic_pairs.csv


In [15]:
# ============================================================
# CELL 12 — 80/10/10 TRAIN / EVAL / TEST SPLIT
# Stratified by label to preserve equal class distribution
# Test set held out completely — never seen during training
# ============================================================
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Step 1: split off 20% as holdout
train_pairs, holdout = train_test_split(
    all_pairs, test_size=0.20,
    stratify=[p['label'] for p in all_pairs],
    random_state=SEED
)

# Step 2: split holdout 50/50 into eval and test
eval_pairs, test_pairs = train_test_split(
    holdout, test_size=0.50,
    stratify=[p['label'] for p in holdout],
    random_state=SEED
)

print(f'Train: {len(train_pairs)} | Eval: {len(eval_pairs)} | Test: {len(test_pairs)}')
for lbl in ['ACCEPT', 'REVISE', 'REJECT']:
    n_tr = sum(1 for p in train_pairs if p['label'] == lbl)
    n_ev = sum(1 for p in eval_pairs  if p['label'] == lbl)
    n_te = sum(1 for p in test_pairs  if p['label'] == lbl)
    print(f'  {lbl}: train={n_tr} eval={n_ev} test={n_te}')

# Save test pairs to Drive for evaluation after training
TEST_PATH = f'{DRIVE_ROOT}/critic_test_pairs.json'
with open(TEST_PATH, 'w') as f:
    json.dump(test_pairs, f, indent=2)
print(f'\nTest pairs saved to Drive: {TEST_PATH}')

train_ds = Dataset.from_list([{'text': p['full_text']} for p in train_pairs])
eval_ds  = Dataset.from_list([{'text': p['full_text']} for p in eval_pairs])
print('Datasets ready.')

Train: 460 | Eval: 58 | Test: 58
  ACCEPT: train=154 eval=19 test=19
  REVISE: train=153 eval=20 test=19
  REJECT: train=153 eval=19 test=20

Test pairs saved to Drive: /content/drive/MyDrive/298B_WB2/critic_test_pairs.json
Datasets ready.


In [16]:
# ============================================================
# CELL 13 — LOAD MODEL AND APPLY LORA
# ============================================================
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print(f'Loading {BASE_MODEL}...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True
)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODS,
    lora_dropout=LORA_DROPOUT,
    bias='none', task_type=TaskType.CAUSAL_LM
))
model.print_trainable_parameters()

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    used_vram  = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM: {used_vram:.1f}GB used / {total_vram:.0f}GB total')

Loading Qwen/Qwen2.5-Coder-7B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 60,555,264 || all params: 7,676,171,776 || trainable%: 0.7889
VRAM: 15.5GB used / 85GB total


In [17]:
# ============================================================
# CELL 14 — TRAINING WITH EARLY STOPPING
#
# Config justification:
#   EPOCHS=6 with EarlyStoppingCallback — model trains until plateau
#   EarlyStoppingCallback patience=3:
#     stops if 3 consecutive evals show no improvement in eval_loss
#     at eval_steps=25 and ~32 steps/epoch this = ~2-3 epochs of plateau
#   This is better than fixed epochs because:
#     576 examples may converge in 2 epochs or need 5 — we don't know yet
#     Early stopping finds the natural convergence point automatically
#
#   load_best_model_at_end=True:
#     after training (or early stop), loads the checkpoint with lowest eval_loss
#     Cell 15 then saves THAT checkpoint as the final adapter
#
#   All checkpoints saved to Drive (OUTPUT_DIR = 298B_WB2/training_run_critic)
#   so even if Colab disconnects mid-training, checkpoints are safe
# ============================================================
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type='cosine',
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    gradient_checkpointing=True,
    eval_strategy='steps', eval_steps=25,
    save_strategy='steps', save_steps=25,
    save_total_limit=4,               # keep 4 checkpoints — enough to cover patience window
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,          # lower eval_loss is better
    logging_steps=10,
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    max_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field='text',
)

# Early stopping: halt if eval_loss doesn't improve for 3 consecutive evals
# At eval_steps=25 and ~32 steps/epoch this fires after ~2-3 epochs of plateau
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,        # 3 consecutive evals without improvement
    early_stopping_threshold=0.001,   # minimum improvement to count as progress
)

trainer = SFTTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=TOKENIZER,
    callbacks=[early_stopping],
)

spe = len(train_pairs) // (BATCH_SIZE * GRAD_ACCUM)
print(f'Train: {len(train_pairs)} | Steps/epoch: ~{spe} | Max steps: ~{spe * EPOCHS}')
print(f'Early stopping: patience=3 evals (~2-3 plateau epochs) | threshold=0.001')
print(f'Checkpoints saved to Drive every 25 steps: {OUTPUT_DIR}')
print('Starting training...')
trainer.train()
print(f'\nTraining complete. Stopped at step {trainer.state.global_step}')
print(f'Best eval loss: {trainer.state.best_metric:.6f} at checkpoint: {trainer.state.best_model_checkpoint}')


Adding EOS to train dataset:   0%|          | 0/460 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/460 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/460 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Train: 460 | Steps/epoch: ~28 | Max steps: ~168
Early stopping: patience=3 evals (~2-3 plateau epochs) | threshold=0.001
Checkpoints saved to Drive every 25 steps: /content/drive/MyDrive/298B_WB2/training_run_critic
Starting training...


Step,Training Loss,Validation Loss
25,1.837657,1.371206
50,1.179939,1.181538
75,1.167040,1.125074
100,0.983274,1.118668
125,0.931054,1.133935
150,0.864039,1.137466



Training complete. Stopped at step 174
Best eval loss: 1.118668 at checkpoint: /content/drive/MyDrive/298B_WB2/training_run_critic/checkpoint-100


In [18]:
# ============================================================
# CELL 15 — SAVE BEST ADAPTER TO DRIVE AND INFERENCE TEST
#
# load_best_model_at_end=True already loaded the best checkpoint
# into model before this cell runs.
# We explicitly save it to ADAPTER_DIR on Drive for persistence.
# ============================================================
import os

# Save best adapter to Drive
model.save_pretrained(ADAPTER_DIR)
TOKENIZER.save_pretrained(ADAPTER_DIR)
print(f'Best adapter saved to Drive: {ADAPTER_DIR}')
print(f'Adapter files:')
for fn in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, fn)) / 1e6
    print(f'  {fn}: {size_mb:.1f}MB')

# Also confirm checkpoints are on Drive
ckpts = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')]
print(f'\nCheckpoints on Drive ({OUTPUT_DIR}):')
for c in sorted(ckpts):
    print(f'  {c}')

# Quick inference test — one of each label
model.eval()
print()
for lbl in ['ACCEPT', 'REVISE', 'REJECT']:
    test = next((p for p in eval_pairs if p['label'] == lbl), None)
    if not test:
        continue
    prompt = f'<|im_start|>user\n{test["input"]}<|im_end|>\n<|im_start|>assistant\n'
    ids    = TOKENIZER.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=150, do_sample=False,
            pad_token_id=TOKENIZER.eos_token_id
        )
    generated = TOKENIZER.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    correct   = f'VERDICT: {lbl}' in generated
    has_reason = 'REASONING:' in generated
    print(f'--- {lbl} TEST (PR #{test["pr_number"]}) ---')
    print(f'Expected:  {test["output"][:200]}')
    print(f'Generated: {generated[:200]}')
    print(f'Verdict: {"PASS" if correct else "FAIL"} | Reasoning: {"PASS" if has_reason else "FAIL"}')
    print()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Best adapter saved to Drive: /content/drive/MyDrive/298B_WB2/critic_lora_adapter
Adapter files:
  README.md: 0.0MB
  adapter_config.json: 0.0MB
  adapter_model.safetensors: 242.3MB
  chat_template.jinja: 0.0MB
  tokenizer.json: 11.4MB
  tokenizer_config.json: 0.0MB

Checkpoints on Drive (/content/drive/MyDrive/298B_WB2/training_run_critic):
  checkpoint-100
  checkpoint-125
  checkpoint-150
  checkpoint-174

--- ACCEPT TEST (PR #56607) ---
Expected:  VERDICT: ACCEPT
REASONING: This patch correctly enhances the `process_executor_events` function by adding a `task_callback_type` parameter that determines the state of the task instance based on its e
Generated: VERDICT: ACCEPT
REASONING: This patch correctly adds a new parameter `task_callback_type` to the function signature, which allows for more precise control over the type of callback being sent. Additio
Verdict: PASS | Reasoning: PASS

--- REVISE TEST (PR #58430) ---
Expected:  VERDICT: REVISE
REASONING: This is the third time we men

In [19]:
# ============================================================
# CELL 16 — LOAD TEST PAIRS AND RUN INFERENCE
# Uses held-out test_pairs saved in Cell 12
# Model is already loaded with best checkpoint from Cell 15
# ============================================================
import json
from pathlib import Path

EVAL_DIR = f'{DRIVE_ROOT}/evals/critic'
os.makedirs(EVAL_DIR, exist_ok=True)

# Load test pairs from Drive
TEST_PATH = f'{DRIVE_ROOT}/critic_test_pairs.json'
with open(TEST_PATH) as f:
    test_pairs = json.load(f)

print(f'Test pairs loaded: {len(test_pairs)}')
for lbl in ['ACCEPT', 'REVISE', 'REJECT']:
    print(f'  {lbl}: {sum(1 for p in test_pairs if p["label"] == lbl)}')


def run_critic_inference(input_text: str) -> str:
    """Run trained Critic model on one input and return raw generated text."""
    encoded    = TOKENIZER(input_text, return_tensors='pt').to(model.device)
    prompt     = f'<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n'
    ids        = TOKENIZER.encode(prompt, return_tensors='pt').to(model.device)
    attn_mask  = torch.ones_like(ids)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            ids,
            attention_mask=attn_mask,
            max_new_tokens=200,
            do_sample=False,
            temperature=1.0,
            pad_token_id=TOKENIZER.eos_token_id,
        )
    return TOKENIZER.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


def extract_verdict(generated: str) -> str:
    """Extract ACCEPT/REVISE/REJECT from generated text. Returns UNKNOWN if not found."""
    for verdict in ['ACCEPT', 'REVISE', 'REJECT']:
        if f'VERDICT: {verdict}' in generated:
            return verdict
    # Fallback: check if verdict word appears anywhere
    text_upper = generated.upper()
    for verdict in ['ACCEPT', 'REVISE', 'REJECT']:
        if verdict in text_upper:
            return verdict
    return 'UNKNOWN'


def extract_reasoning(generated: str) -> str:
    """Extract reasoning text after REASONING: marker."""
    if 'REASONING:' in generated:
        return generated.split('REASONING:')[1].strip()[:500]
    return generated.strip()[:500]


# Run inference on all test pairs
print('\nRunning inference on test pairs...')
inference_results = []

for i, pair in enumerate(test_pairs):
    generated       = run_critic_inference(pair['input'])
    predicted_label = extract_verdict(generated)
    reasoning       = extract_reasoning(generated)

    inference_results.append({
        'pr_number':       pair.get('pr_number'),
        'true_label':      pair['label'],
        'predicted_label': predicted_label,
        'correct':         predicted_label == pair['label'],
        'generated':       generated,
        'reasoning':       reasoning,
        'reference_output': pair['output'],
        'input':           pair['input'],
    })

    if (i + 1) % 10 == 0:
        correct_so_far = sum(1 for r in inference_results if r['correct'])
        print(f'  {i+1}/{len(test_pairs)} | accuracy so far: {correct_so_far/(i+1)*100:.1f}%')

overall_acc = sum(1 for r in inference_results if r['correct']) / len(inference_results)
print(f'\nInference complete. Overall accuracy: {overall_acc*100:.1f}%')

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test pairs loaded: 58
  ACCEPT: 19
  REVISE: 19
  REJECT: 20

Running inference on test pairs...
  10/58 | accuracy so far: 70.0%
  20/58 | accuracy so far: 65.0%
  30/58 | accuracy so far: 63.3%
  40/58 | accuracy so far: 62.5%
  50/58 | accuracy so far: 66.0%

Inference complete. Overall accuracy: 63.8%


In [20]:
# ============================================================
# CELL 17 — F1 CLASSIFICATION METRICS AND PLOTS
#
# Computes per-class and macro F1, precision, recall
# Plots:
#   1. Confusion matrix heatmap
#   2. Per-class precision / recall / F1 bar chart
#   3. Per-class accuracy comparison
# All saved to Drive: evals/critic/
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict

LABELS = ['ACCEPT', 'REVISE', 'REJECT']
LABEL_IDX = {l: i for i, l in enumerate(LABELS)}

# ---- Build confusion matrix ----
cm = np.zeros((3, 3), dtype=int)
for r in inference_results:
    true_idx = LABEL_IDX.get(r['true_label'], -1)
    pred_idx = LABEL_IDX.get(r['predicted_label'], -1)
    if true_idx >= 0 and pred_idx >= 0:
        cm[true_idx][pred_idx] += 1

# ---- Compute per-class metrics ----
metrics = {}
for i, lbl in enumerate(LABELS):
    TP = cm[i][i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - TP - FP - FN
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy  = (TP + TN) / cm.sum()
    metrics[lbl] = {'precision': precision, 'recall': recall, 'f1': f1,
                    'accuracy': accuracy, 'TP': int(TP), 'FP': int(FP),
                    'FN': int(FN), 'TN': int(TN)}

# Macro averages
macro_precision = np.mean([metrics[l]['precision'] for l in LABELS])
macro_recall    = np.mean([metrics[l]['recall']    for l in LABELS])
macro_f1        = np.mean([metrics[l]['f1']        for l in LABELS])
overall_acc     = sum(1 for r in inference_results if r['correct']) / len(inference_results)

print('=' * 55)
print('CLASSIFICATION METRICS')
print('=' * 55)
print(f'{"Label":10} {"Precision":12} {"Recall":10} {"F1":10} {"TP":5} {"FP":5} {"FN":5}')
print('-' * 55)
for lbl in LABELS:
    m = metrics[lbl]
    print(f'{lbl:10} {m["precision"]:12.3f} {m["recall"]:10.3f} {m["f1"]:10.3f} {m["TP"]:5} {m["FP"]:5} {m["FN"]:5}')
print('-' * 55)
print(f'{"MACRO":10} {macro_precision:12.3f} {macro_recall:10.3f} {macro_f1:10.3f}')
print(f'Overall accuracy: {overall_acc:.3f}')

# ============================================================
# PLOT 1 — CONFUSION MATRIX
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Critic Model v1 — Classification Evaluation', fontsize=14, fontweight='bold')

ax1 = axes[0]
im  = ax1.imshow(cm, cmap='Blues')
ax1.set_xticks(range(3))
ax1.set_yticks(range(3))
ax1.set_xticklabels([f'Pred\n{l}' for l in LABELS], fontsize=9)
ax1.set_yticklabels([f'True {l}' for l in LABELS], fontsize=9)
ax1.set_title(f'Confusion Matrix\nOverall Acc={overall_acc:.3f}')
plt.colorbar(im, ax=ax1)
for i in range(3):
    for j in range(3):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax1.text(j, i, str(cm[i, j]), ha='center', va='center',
                 fontsize=14, fontweight='bold', color=color)

# ============================================================
# PLOT 2 — PER-CLASS PRECISION / RECALL / F1
# ============================================================
ax2    = axes[1]
x      = np.arange(3)
width  = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']

b1 = ax2.bar(x - width, [metrics[l]['precision'] for l in LABELS], width, label='Precision', color=colors[0], alpha=0.85)
b2 = ax2.bar(x,          [metrics[l]['recall']    for l in LABELS], width, label='Recall',    color=colors[1], alpha=0.85)
b3 = ax2.bar(x + width,  [metrics[l]['f1']        for l in LABELS], width, label='F1',        color=colors[2], alpha=0.85)

ax2.set_ylim(0, 1.15)
ax2.set_xticks(x)
ax2.set_xticklabels(LABELS)
ax2.set_ylabel('Score')
ax2.set_title(f'Per-Class Metrics\nMacro F1={macro_f1:.3f}')
ax2.axhline(y=0.7, color='green', linestyle='--', alpha=0.4, label='0.7 threshold')
ax2.legend(fontsize=8)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                 f'{h:.2f}', ha='center', va='bottom', fontsize=7)

# ============================================================
# PLOT 3 — PER-CLASS ACCURACY BREAKDOWN
# ============================================================
ax3 = axes[2]
per_class_acc = []
per_class_n   = []
for lbl in LABELS:
    subset = [r for r in inference_results if r['true_label'] == lbl]
    acc    = sum(1 for r in subset if r['correct']) / len(subset) if subset else 0
    per_class_acc.append(acc)
    per_class_n.append(len(subset))

bar_colors = ['#2196F3' if a >= 0.7 else '#F44336' for a in per_class_acc]
bars3 = ax3.bar(LABELS, per_class_acc, color=bar_colors, alpha=0.85, edgecolor='white')
ax3.set_ylim(0, 1.15)
ax3.set_ylabel('Accuracy')
ax3.set_title('Per-Class Accuracy\n(blue=≥0.7, red=<0.7)')
ax3.axhline(y=0.7, color='green', linestyle='--', alpha=0.5)

for bar, acc, n in zip(bars3, per_class_acc, per_class_n):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.2f}\n(n={n})', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
f1_plot_path = f'{EVAL_DIR}/critic_f1_metrics.png'
plt.savefig(f1_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'F1 plot saved: {f1_plot_path}')

CLASSIFICATION METRICS
Label      Precision    Recall     F1         TP    FP    FN   
-------------------------------------------------------
ACCEPT            0.760      1.000      0.864    19     6     0
REVISE            0.600      0.632      0.615    12     8     7
REJECT            0.462      0.300      0.364     6     7    14
-------------------------------------------------------
MACRO             0.607      0.644      0.614
Overall accuracy: 0.638
F1 plot saved: /content/drive/MyDrive/298B_WB2/evals/critic/critic_f1_metrics.png


In [21]:
# ============================================================
# CELL 18 — G-EVAL REASONING QUALITY
#
# G-Eval evaluates the REASONING text quality on 4 dimensions:
#
# 1. VERDICT_ACCURACY (1-5)
#    Is ACCEPT/REVISE/REJECT the right call given the diff and plan?
#    Evaluated against the reference output (ground truth label)
#
# 2. REASONING_SPECIFICITY (1-5)
#    Does the reasoning reference specific aspects of the diff?
#    (file names, function names, specific code patterns)
#    vs generic phrases like 'looks good' or 'needs improvement'
#
# 3. REASONING_ACCURACY (1-5)
#    Is the reasoning factually correct about what the diff does?
#    Does it avoid hallucinating code details not present in the diff?
#
# 4. ACTIONABILITY (1-5)
#    For REVISE/REJECT: does the reasoning tell the Patcher
#    exactly what to fix? For ACCEPT: does it confirm what was done well?
#    A Patcher receiving this feedback could act on it immediately.
#
# Run on ALL test pairs (both correct and incorrect verdicts)
# Incorrect verdicts will score low on VERDICT_ACCURACY — that's expected
# and gives a full picture of model quality
# ============================================================
import time

GEVAL_PROMPT = """You are evaluating the output of a code review model (the Critic)
in an agentic AI system for Apache Airflow patch generation.

The Critic receives a patch plan and diff, then outputs ACCEPT, REVISE, or REJECT
with reasoning. Its output is consumed by downstream models — REVISE/REJECT feedback
goes back to the Patcher for revision, ACCEPT triggers sandbox testing.

PLAN AND DIFF (what the Critic saw as input):
{input_text}

CRITIC OUTPUT (what the model generated):
Verdict: {predicted_verdict}
Reasoning: {generated_reasoning}

GROUND TRUTH (correct verdict and reference reasoning):
{reference_output}

Evaluate on these four dimensions. Score 1-5 each.

1. VERDICT_ACCURACY (1-5)
   Does the model's ACCEPT/REVISE/REJECT match what the ground truth says?
   5 = correct verdict matching ground truth
   3 = adjacent error (REVISE instead of REJECT or vice versa — less severe)
   1 = completely wrong (ACCEPT when should be REJECT, or vice versa)
   Note: REVISE vs REJECT confusion is less severe than ACCEPT vs REJECT confusion.

2. REASONING_SPECIFICITY (1-5)
   Does the reasoning reference specific elements of the diff?
   5 = references specific file names, function names, or exact code patterns from the diff
   3 = references the general area of the change but not specific details
   1 = completely generic ('looks good', 'needs improvement', 'code quality issue')

3. REASONING_ACCURACY (1-5)
   Is the reasoning factually correct about what the diff actually does?
   5 = all claims about the code are accurate, nothing invented
   3 = mostly accurate, minor inaccuracy or slight overstatement
   1 = hallucinates code details, describes changes not present in the diff

4. ACTIONABILITY (1-5)
   Could the Patcher immediately act on this feedback to improve the patch?
   For REVISE/REJECT: 5 = tells Patcher exactly what to change and why
   For ACCEPT: 5 = clearly confirms what was done correctly
   3 = directionally helpful but vague on specifics
   1 = Patcher would not know what to do with this feedback

Output exactly this JSON and nothing else — no preamble, no explanation outside the JSON:
{{
  "verdict_accuracy": <1-5>,
  "reasoning_specificity": <1-5>,
  "reasoning_accuracy": <1-5>,
  "actionability": <1-5>,
  "reasoning": "<one sentence identifying the single biggest weakness in this output>"
}}
"""


def geval_critic(result: dict, max_retries: int = 5) -> dict:
    """Score one Critic output with G-Eval using GPT-4o."""
    prompt = GEVAL_PROMPT.format(
        input_text=result['input'][:1200],
        predicted_verdict=result['predicted_label'],
        generated_reasoning=result['reasoning'][:400],
        reference_output=result['reference_output'][:300],
    )
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model='gpt-4o',
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0,
            )
            raw = resp.choices[0].message.content.strip()
            # Strip markdown code fences if present
            raw = re.sub(r'^```json\s*', '', raw)
            raw = re.sub(r'```$', '', raw).strip()
            return json.loads(raw)
        except openai.RateLimitError:
            wait = (2 ** attempt) * 5 + random.uniform(0, 3)
            print(f'  Rate limit (attempt {attempt+1}/{max_retries}) — waiting {wait:.1f}s')
            time.sleep(wait)
        except openai.APIStatusError as e:
            if e.status_code == 429:
                wait = (2 ** attempt) * 5 + random.uniform(0, 3)
                print(f'  429 (attempt {attempt+1}/{max_retries}) — waiting {wait:.1f}s')
                time.sleep(wait)
            else:
                print(f'  API error {e.status_code}')
                return {}
        except Exception as e:
            match = re.search(r'\{.*\}', resp.choices[0].message.content if 'resp' in dir() else '', re.DOTALL)
            if match:
                try: return json.loads(match.group())
                except: pass
            print(f'  Parse error: {e}')
            return {}
    return {}


print(f'Running G-Eval on {len(inference_results)} test pairs...')
print('Using GPT-4o — 2s delay between calls to respect TPM limits')
print()

geval_results = []
GEVAL_DIMS = ['verdict_accuracy', 'reasoning_specificity', 'reasoning_accuracy', 'actionability']

for i, result in enumerate(inference_results):
    scores = geval_critic(result)
    time.sleep(2)  # proactive pacing

    geval_entry = {
        'pr_number':       result['pr_number'],
        'true_label':      result['true_label'],
        'predicted_label': result['predicted_label'],
        'correct':         result['correct'],
        'generated':       result['generated'][:400],
        **scores,
    }
    geval_results.append(geval_entry)

    if scores:
        avg_so_far = {d: sum(r.get(d,0) for r in geval_results if r.get(d)) /
                         max(sum(1 for r in geval_results if r.get(d)), 1)
                     for d in GEVAL_DIMS}
        print(
            f'[{i+1:2}/{len(inference_results)}] PR #{result["pr_number"]} '
            f'true={result["true_label"]} pred={result["predicted_label"]} '
            f'VA={scores.get("verdict_accuracy","?")} '
            f'RS={scores.get("reasoning_specificity","?")} '
            f'RA={scores.get("reasoning_accuracy","?")} '
            f'AC={scores.get("actionability","?")}'
        )

print('\nG-Eval complete.')

Running G-Eval on 58 test pairs...
Using GPT-4o — 2s delay between calls to respect TPM limits

[ 1/58] PR #59791 true=REVISE pred=REVISE VA=5 RS=3 RA=4 AC=4
[ 2/58] PR #60367 true=ACCEPT pred=ACCEPT VA=5 RS=3 RA=4 AC=5
[ 3/58] PR #57398 true=ACCEPT pred=ACCEPT VA=5 RS=3 RA=5 AC=5
[ 4/58] PR #62896 true=REVISE pred=REJECT VA=3 RS=1 RA=5 AC=1
[ 5/58] PR #59882 true=REJECT pred=REJECT VA=5 RS=3 RA=4 AC=2
[ 6/58] PR #58152 true=ACCEPT pred=ACCEPT VA=5 RS=1 RA=1 AC=1
[ 7/58] PR #63677 true=REJECT pred=ACCEPT VA=1 RS=1 RA=1 AC=1
[ 8/58] PR #58368 true=REVISE pred=REVISE VA=5 RS=1 RA=5 AC=3
[ 9/58] PR #62432 true=ACCEPT pred=ACCEPT VA=5 RS=3 RA=4 AC=5
[10/58] PR #61778 true=REJECT pred=REVISE VA=3 RS=3 RA=5 AC=3
[11/58] PR #55643 true=REVISE pred=REJECT VA=3 RS=1 RA=3 AC=1
[12/58] PR #59389 true=REJECT pred=REVISE VA=3 RS=1 RA=3 AC=2
[13/58] PR #60977 true=ACCEPT pred=ACCEPT VA=5 RS=3 RA=3 AC=5
[14/58] PR #61034 true=REJECT pred=ACCEPT VA=1 RS=3 RA=2 AC=1
[15/58] PR #58646 true=REVISE pred=R

In [22]:
# ============================================================
# CELL 18B — G-EVAL AGGREGATED SUMMARY
# Run after Cell 18 completes — uses geval_results in memory
# ============================================================

GEVAL_DIMS = ['verdict_accuracy', 'reasoning_specificity', 'reasoning_accuracy', 'actionability']
LABELS     = ['ACCEPT', 'REVISE', 'REJECT']

valid_geval = [r for r in geval_results if r.get('verdict_accuracy')]
print(f'Valid G-Eval results: {len(valid_geval)} / {len(geval_results)}')

# ---- Overall averages ----
print()
print('=' * 58)
print('G-EVAL OVERALL AVERAGES')
print('=' * 58)
for d in GEVAL_DIMS:
    vals = [r.get(d, 0) for r in valid_geval]
    avg  = sum(vals) / len(vals)
    mn   = min(vals)
    mx   = max(vals)
    dist = {s: vals.count(s) for s in range(1, 6)}
    print(f'  {d:28}: avg={avg:.2f}  min={mn}  max={mx}  dist={dist}')

# ---- Per-label breakdown ----
print()
print('=' * 58)
print('G-EVAL BY TRUE LABEL')
print('=' * 58)
for lbl in LABELS:
    subset = [r for r in valid_geval if r['true_label'] == lbl]
    if not subset:
        continue
    print(f'\n  {lbl} (n={len(subset)}):')
    for d in GEVAL_DIMS:
        vals = [r.get(d, 0) for r in subset]
        avg  = sum(vals) / len(vals)
        print(f'    {d:28}: {avg:.2f}')

# ---- Correct vs incorrect verdict G-Eval comparison ----
print()
print('=' * 58)
print('G-EVAL: CORRECT VERDICTS vs INCORRECT VERDICTS')
print('=' * 58)
correct_set   = [r for r in valid_geval if r['correct']]
incorrect_set = [r for r in valid_geval if not r['correct']]
print(f'  Correct predictions:   {len(correct_set)}')
print(f'  Incorrect predictions: {len(incorrect_set)}')
print()
for d in GEVAL_DIMS:
    c_avg = sum(r.get(d,0) for r in correct_set)   / max(len(correct_set), 1)
    i_avg = sum(r.get(d,0) for r in incorrect_set) / max(len(incorrect_set), 1)
    print(f'  {d:28}: correct={c_avg:.2f}  incorrect={i_avg:.2f}')

# ---- Weakest reasoning examples ----
print()
print('=' * 58)
print('5 WEAKEST EXAMPLES (lowest actionability)')
print('=' * 58)
weakest = sorted(valid_geval, key=lambda r: r.get('actionability', 5))[:5]
for r in weakest:
    print(f'  PR #{r["pr_number"]} | true={r["true_label"]} pred={r["predicted_label"]} | '
          f'AC={r.get("actionability")} VA={r.get("verdict_accuracy")}')
    print(f'  Weakness: {r.get("reasoning", "n/a")}')
    print()

Valid G-Eval results: 58 / 58

G-EVAL OVERALL AVERAGES
  verdict_accuracy            : avg=4.07  min=1  max=5  dist={1: 6, 2: 0, 3: 15, 4: 0, 5: 37}
  reasoning_specificity       : avg=2.45  min=1  max=5  dist={1: 24, 2: 0, 3: 23, 4: 6, 5: 5}
  reasoning_accuracy          : avg=3.69  min=1  max=5  dist={1: 9, 2: 1, 3: 14, 4: 9, 5: 25}
  actionability               : avg=2.93  min=1  max=5  dist={1: 17, 2: 8, 3: 12, 4: 4, 5: 17}

G-EVAL BY TRUE LABEL

  ACCEPT (n=19):
    verdict_accuracy            : 5.00
    reasoning_specificity       : 3.11
    reasoning_accuracy          : 3.26
    actionability               : 4.37

  REVISE (n=19):
    verdict_accuracy            : 4.26
    reasoning_specificity       : 1.63
    reasoning_accuracy          : 4.32
    actionability               : 2.47

  REJECT (n=20):
    verdict_accuracy            : 3.00
    reasoning_specificity       : 2.60
    reasoning_accuracy          : 3.50
    actionability               : 2.00

G-EVAL: CORRECT VERDICT

In [23]:
# ============================================================
# CELL 19 — G-EVAL PLOTS AND SAVE ALL RESULTS TO DRIVE
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

valid_geval = [r for r in geval_results if r.get('verdict_accuracy')]

# ---- Print G-Eval summary ----
print('=' * 55)
print('G-EVAL REASONING QUALITY SUMMARY')
print('=' * 55)
for d in GEVAL_DIMS:
    vals = [r.get(d, 0) for r in valid_geval]
    print(f'  {d:28}: avg={sum(vals)/len(vals):.2f} min={min(vals)} max={max(vals)}')

# Per-label breakdown
print()
for lbl in LABELS:
    subset = [r for r in valid_geval if r['true_label'] == lbl]
    if not subset: continue
    print(f'  {lbl} (n={len(subset)}):')
    for d in GEVAL_DIMS:
        vals = [r.get(d, 0) for r in subset]
        print(f'    {d:28}: {sum(vals)/len(vals):.2f}')

# ============================================================
# PLOT 1 — G-EVAL DIMENSION AVERAGES (overall + per label)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Critic Model v1 — G-Eval Reasoning Quality', fontsize=14, fontweight='bold')

# Overall averages
ax1    = axes[0]
dim_labels = ['Verdict\nAccuracy', 'Reasoning\nSpecificity', 'Reasoning\nAccuracy', 'Actionability']
overall_avgs = [sum(r.get(d, 0) for r in valid_geval) / len(valid_geval) for d in GEVAL_DIMS]
colors_dim = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
bars = ax1.bar(dim_labels, overall_avgs, color=colors_dim, alpha=0.85, edgecolor='white')
ax1.set_ylim(0, 5.5)
ax1.set_ylabel('Score (1-5)')
ax1.set_title('Overall G-Eval Averages')
ax1.axhline(y=3.5, color='green', linestyle='--', alpha=0.5, label='Good (3.5)')
ax1.axhline(y=4.0, color='blue',  linestyle='--', alpha=0.5, label='Strong (4.0)')
ax1.legend(fontsize=8)
for bar, val in zip(bars, overall_avgs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

# Per-label breakdown for Verdict Accuracy and Actionability
ax2   = axes[1]
x     = np.arange(len(LABELS))
w     = 0.2
label_colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for di, (d, color) in enumerate(zip(GEVAL_DIMS, label_colors)):
    per_lbl_avgs = []
    for lbl in LABELS:
        subset = [r for r in valid_geval if r['true_label'] == lbl]
        avg    = sum(r.get(d, 0) for r in subset) / len(subset) if subset else 0
        per_lbl_avgs.append(avg)
    offset = (di - 1.5) * w
    bars2  = ax2.bar(x + offset, per_lbl_avgs, w, label=d.replace('_', ' ').title(),
                     color=color, alpha=0.8)

ax2.set_ylim(0, 5.5)
ax2.set_xticks(x)
ax2.set_xticklabels(LABELS)
ax2.set_ylabel('Score (1-5)')
ax2.set_title('G-Eval Scores by True Label')
ax2.axhline(y=3.5, color='green', linestyle='--', alpha=0.4)
ax2.legend(fontsize=7, loc='lower right')

plt.tight_layout()
geval_plot_path = f'{EVAL_DIR}/critic_geval_scores.png'
plt.savefig(geval_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'G-Eval plot saved: {geval_plot_path}')

# ============================================================
# PLOT 2 — CORRECT vs INCORRECT BY LABEL WITH G-EVAL OVERLAY
# ============================================================
fig2, ax3 = plt.subplots(figsize=(10, 5))
fig2.suptitle('Verdict Correctness vs G-Eval Reasoning Quality', fontsize=12, fontweight='bold')

x      = np.arange(len(LABELS))
w      = 0.35
acc_vals = []
va_vals  = []

for lbl in LABELS:
    subset = [r for r in valid_geval if r['true_label'] == lbl]
    acc    = sum(1 for r in subset if r['correct']) / len(subset) if subset else 0
    va     = sum(r.get('verdict_accuracy', 0) for r in subset) / len(subset) if subset else 0
    acc_vals.append(acc)
    va_vals.append(va / 5)  # normalise to 0-1 for comparison

b1 = ax3.bar(x - w/2, acc_vals, w, label='Classification Accuracy', color='#2196F3', alpha=0.85)
b2 = ax3.bar(x + w/2, va_vals,  w, label='G-Eval Verdict Accuracy (normalised)', color='#FF9800', alpha=0.85)
ax3.set_ylim(0, 1.2)
ax3.set_xticks(x)
ax3.set_xticklabels(LABELS)
ax3.set_ylabel('Score (0-1)')
ax3.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='0.7 threshold')
ax3.legend()

for bars, vals in [(b1, acc_vals), (b2, va_vals)]:
    for bar, val in zip(bars, vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
overlay_plot_path = f'{EVAL_DIR}/critic_correctness_vs_geval.png'
plt.savefig(overlay_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Overlay plot saved: {overlay_plot_path}')

# ============================================================
# SAVE ALL RESULTS JSON TO DRIVE
# ============================================================
results_path = f'{EVAL_DIR}/critic_eval_results.json'
with open(results_path, 'w') as f:
    json.dump({
        'inference_results': inference_results,
        'geval_results':     geval_results,
        'classification_metrics': metrics,
        'macro_f1':          macro_f1,
        'macro_precision':   macro_precision,
        'macro_recall':      macro_recall,
        'overall_accuracy':  overall_acc,
        'geval_dim_avgs':    {d: sum(r.get(d,0) for r in valid_geval)/len(valid_geval)
                              for d in GEVAL_DIMS},
        'adapter_dir':       ADAPTER_DIR,
        'n_test':            len(inference_results),
    }, f, indent=2)

print(f'\nAll results saved to Drive: {EVAL_DIR}')
print(f'  critic_eval_results.json')
print(f'  critic_f1_metrics.png')
print(f'  critic_geval_scores.png')
print(f'  critic_correctness_vs_geval.png')
print(f'\nSummary:')
print(f'  Overall accuracy:  {overall_acc:.3f}')
print(f'  Macro F1:          {macro_f1:.3f}')
for d in GEVAL_DIMS:
    vals = [r.get(d,0) for r in valid_geval]
    print(f'  {d:28}: {sum(vals)/len(vals):.2f}')

G-EVAL REASONING QUALITY SUMMARY
  verdict_accuracy            : avg=4.07 min=1 max=5
  reasoning_specificity       : avg=2.45 min=1 max=5
  reasoning_accuracy          : avg=3.69 min=1 max=5
  actionability               : avg=2.93 min=1 max=5

  ACCEPT (n=19):
    verdict_accuracy            : 5.00
    reasoning_specificity       : 3.11
    reasoning_accuracy          : 3.26
    actionability               : 4.37
  REVISE (n=19):
    verdict_accuracy            : 4.26
    reasoning_specificity       : 1.63
    reasoning_accuracy          : 4.32
    actionability               : 2.47
  REJECT (n=20):
    verdict_accuracy            : 3.00
    reasoning_specificity       : 2.60
    reasoning_accuracy          : 3.50
    actionability               : 2.00
G-Eval plot saved: /content/drive/MyDrive/298B_WB2/evals/critic/critic_geval_scores.png
Overlay plot saved: /content/drive/MyDrive/298B_WB2/evals/critic/critic_correctness_vs_geval.png

All results saved to Drive: /content/drive/MyDriv